# Análisis columna por columna — train.csv
Ejecuta cada celda en orden. Cada sección tiene celdas de exploración y luego la limpieza decidida.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('train__1_.csv')
print('Shape:', df.shape)
print()
print('Vista general de nulos:')
print(df.isnull().sum())
print()
print('Tipos de datos:')
print(df.dtypes)

---
## Columna: `id`
**Decisión: eliminar** — identificador único sin valor predictivo.

In [ ]:
col = 'id'

print(f'Nulos: {df[col].isnull().sum()}')
print(f'Únicos: {df[col].nunique()} de {len(df)} filas')
print(f'¿Todos únicos?: {df[col].nunique() == len(df)}')
print()
print('Muestra:')
print(df[col].head(5).tolist())

In [ ]:
# LIMPIEZA
df = df.drop(columns=['id'])
print('Columna id eliminada.')
print('Shape:', df.shape)

---
## Columna: `label`
**Decisión: sin limpieza.** Desbalance moderado (1.84x). Se tratará al modelar solo si el recall de clase 0 es bajo.

In [ ]:
col = 'label'

print(f'Nulos: {df[col].isnull().sum()}')
print(f'Tipo: {df[col].dtype}')
print()
print('Distribución de clases:')
vc = df[col].value_counts()
for val, count in vc.items():
    print(f'  {val}: {count} ({count/len(df)*100:.1f}%)')
print()
print(f'Ratio desbalance (mayoritaria/minoritaria): {vc.max()/vc.min():.2f}x')

In [ ]:
# Sin limpieza en esta columna.
# Si al entrenar el recall de clase 0 es bajo, activar class_weight:
#
# from sklearn.utils.class_weight import compute_class_weight
# weights = compute_class_weight('balanced', classes=[0,1], y=df['label'])
# class_weight = dict(zip([0,1], weights))
# → pasar al modelo: LogisticRegression(class_weight=class_weight)

print('Sin cambios en label.')

---
## Columna: `statement`
**Decisión:** strip + eliminar paréntesis (anotaciones del transcriptor) + eliminar duplicados exactos (mismo statement + speaker + label).

In [ ]:
col = 'statement'

print(f'Nulos: {df[col].isnull().sum()}')
print(f'Únicos: {df[col].nunique()} de {len(df)}')
print()
leading = df[col].str.startswith(' ').sum()
trailing = df[col].str.endswith(' ').sum()
print(f'Con espacio al inicio: {leading}')
print(f'Con espacio al final:  {trailing}')

In [ ]:
# Ver statements duplicados y si tienen distinto label/speaker
dups = df[df.duplicated(subset=col, keep=False)].sort_values(col)
print(f'Filas con statement duplicado: {len(dups)}')
print()
print(dups[['statement', 'speaker', 'label']].to_string())

In [ ]:
# Caso más problemático: mismo statement + mismo speaker, labels distintos
dup_conflict = df[df.duplicated(subset=[col, 'speaker'], keep=False)].sort_values([col, 'speaker'])
print(f'Duplicados exactos (statement + speaker): {len(dup_conflict)}')
print()
print(dup_conflict[['statement', 'speaker', 'label']].to_string())

In [ ]:
# Ver ejemplos con paréntesis ANTES de limpiar
mask = df[col].str.contains(r'\(.*?\)', regex=True)
print(f'Filas con paréntesis: {mask.sum()}')
print()
print(df[mask][col].head(10).to_string())

In [ ]:
# LIMPIEZA

# Paso 1: quitar espacios
df[col] = df[col].str.strip()
print('Espacios eliminados.')

# Paso 2: quitar anotaciones entre paréntesis (las añadió el transcriptor, no el speaker)
df[col] = df[col].str.replace(r'\(.*?\)', '', regex=True).str.strip()
print('Paréntesis eliminados.')

# Paso 3: eliminar duplicados exactos (mismo statement + speaker + label)
# Los que tienen distinto speaker o distinto label NO se eliminan: son registros legítimos
antes = len(df)
df = df.drop_duplicates(subset=[col, 'speaker', 'label'])
print(f'Duplicados exactos eliminados: {antes - len(df)} filas')
print(f'Shape final: {df.shape}')

---
## Columna: `subject`
**Decisión:** crear columna `subject_primary` con solo el primer subject. La columna original se mantiene.

In [ ]:
col = 'subject'

print(f'Nulos: {df[col].isnull().sum()}')
print(f'Valores únicos (combinaciones): {df[col].nunique()}')
print()
print('Top 10 combinaciones más frecuentes:')
print(df[col].value_counts().head(10))

In [ ]:
# Explotar por comas para ver subjects individuales
all_subjects = df[col].str.split(',').explode().str.strip()
print(f'Subjects individuales únicos: {all_subjects.nunique()}')
print()
print('Top 15 subjects individuales:')
print(all_subjects.value_counts().head(15))

In [ ]:
# Cuántos subjects tiene cada fila
n_subjects = df[col].str.split(',').apply(len)
print('Número de subjects por fila:')
print(n_subjects.describe())
print()
print(f'Filas con 1 solo subject: {(n_subjects == 1).sum()}')
print(f'Filas con más de 3 subjects: {(n_subjects > 3).sum()}')

In [ ]:
# LIMPIEZA
df['subject_primary'] = df[col].str.split(',').str[0].str.strip()
print('Nueva columna subject_primary creada.')
print()
print('Top 10 valores:')
print(df['subject_primary'].value_counts().head(10))
print(f'Únicos: {df["subject_primary"].nunique()}')

---
## Columna: `speaker`
**Decisión:** umbral 5 — agrupar speakers con menos de 5 apariciones como `'other'` en nueva columna `speaker_grouped`. El 90% de speakers únicos tienen 5 o menos apariciones.

In [ ]:
col = 'speaker'

print(f'Nulos: {df[col].isnull().sum()}')
print(f'Únicos: {df[col].nunique()}')
print()
print('Top 10 speakers:')
print(df[col].value_counts().head(10))

In [ ]:
# Distribución de frecuencias bajas
vc = df[col].value_counts()
for n in [1, 2, 3, 4, 5]:
    print(f'Speakers con {n} aparición/es: {(vc == n).sum()}')

In [ ]:
# Impacto de distintos umbrales (para justificar la elección)
for umbral in [5, 10, 20, 50]:
    vc = df[col].value_counts()
    frecuentes = vc[vc >= umbral].index
    agrupados = (~df[col].isin(frecuentes)).sum()
    print(f'Umbral {umbral:>3}: {len(frecuentes):>4} speakers frecuentes | {agrupados:>5} filas como "other" ({agrupados/len(df)*100:.1f}%)')

In [ ]:
# LIMPIEZA: umbral 5
MIN_COUNT = 5
vc = df[col].value_counts()
speakers_frecuentes = vc[vc >= MIN_COUNT].index
df['speaker_grouped'] = df[col].where(df[col].isin(speakers_frecuentes), other='other')

print(f'Speakers frecuentes (>={MIN_COUNT}): {len(speakers_frecuentes)}')
print(f'Agrupados como "other": {(df["speaker_grouped"] == "other").sum()} filas ({(df["speaker_grouped"] == "other").sum()/len(df)*100:.1f}%)')
print(f'Únicos en speaker_grouped: {df["speaker_grouped"].nunique()}')

---
## Columna: `speaker_job`
**Decisión:** normalizar capitalización (lower+strip) + rellenar nulos con `'unknown'`. Los nulos no son aleatorios: se concentran en perfiles sin cargo definido.

In [ ]:
col = 'speaker_job'

print(f'Nulos: {df[col].isnull().sum()} ({df[col].isnull().mean()*100:.1f}%)')
print(f'Únicos (tal cual): {df[col].nunique()}')
print()
print('Top 10 valores:')
print(df[col].value_counts().head(10))

In [ ]:
# Detectar duplicados por capitalización
print(f'Únicos sin normalizar: {df[col].nunique()}')
print(f'Únicos normalizados (lower+strip): {df[col].str.lower().str.strip().nunique()}')
print()
print('Top 10 en minúsculas:')
print(df[col].str.lower().str.strip().value_counts().head(10))

In [ ]:
# Patrón de los nulos
print('Nulos por party_affiliation:')
print(df[df[col].isnull()]['party_affiliation'].value_counts())

In [ ]:
# LIMPIEZA

# Paso 1: normalizar capitalización
df[col] = df[col].str.lower().str.strip()
print(f'Únicos tras normalizar: {df[col].nunique()}')

# Paso 2: rellenar nulos con 'unknown'
# Motivo: concentrados en perfiles sin cargo real (none, organization...)
# Imputar con la moda sería inventar información
df[col] = df[col].fillna('unknown')
print(f'Nulos restantes: {df[col].isnull().sum()}')
print()
print('Top 10 tras limpieza:')
print(df[col].value_counts().head(10))

In [ ]:
# Opcional: ver variantes similares que podrían fusionarse
mask = df[col].str.contains('senator', na=False)
print(f'Variantes de senator: {df[mask][col].nunique()}')
print(df[mask][col].value_counts())

---
## Columna: `state_info`
**Decisión:** rellenar nulos con `'unknown'`. El 55% de los nulos vienen de party='none' — no son datos faltantes por error, esas personas no tienen estado asociado. Imputar con la moda sería inventar información.

In [ ]:
col = 'state_info'

print(f'Nulos: {df[col].isnull().sum()} ({df[col].isnull().mean()*100:.1f}%)')
print(f'Únicos: {df[col].nunique()}')
print()
print('Top 10 estados:')
print(df[col].value_counts().head(10))

In [ ]:
# Verificar que los nulos tienen un patrón
print('Nulos por party_affiliation:')
print(df[df[col].isnull()]['party_affiliation'].value_counts())
print()
nulos_por_party = df.groupby('party_affiliation')[col].apply(lambda x: x.isnull().mean() * 100).sort_values(ascending=False)
print('% nulos por party_affiliation:')
print(nulos_por_party.round(1))

In [ ]:
# LIMPIEZA
df[col] = df[col].fillna('unknown')
print(f'Nulos tras fillna: {df[col].isnull().sum()}')
print(f'Únicos ahora (incluye unknown): {df[col].nunique()}')

---
## Columna: `party_affiliation`
**Decisión:** agrupar partidos con menos de 50 apariciones como `'other'` en nueva columna `party_grouped`.

In [ ]:
col = 'party_affiliation'

print(f'Nulos: {df[col].isnull().sum()}')
print(f'Únicos: {df[col].nunique()}')
print()
print('Distribución completa:')
vc = df[col].value_counts()
for val, count in vc.items():
    print(f'  {val:<25} {count:>5}  ({count/len(df)*100:.1f}%)')

In [ ]:
# Ver categorías minoritarias
MIN_FREQ = 50
vc = df[col].value_counts()
minoritarias = vc[vc < MIN_FREQ].index
print(f'Categorías con menos de {MIN_FREQ} apariciones ({len(minoritarias)} partidos):')
print(vc[vc < MIN_FREQ])

In [ ]:
# LIMPIEZA
df['party_grouped'] = df[col].where(~df[col].isin(minoritarias), other='other')
print('Distribución tras agrupar minoritarias:')
print(df['party_grouped'].value_counts())

---
## Resumen final del dataset limpio

In [ ]:
print('Shape final:', df.shape)
print()
print('Nulos por columna:')
print(df.isnull().sum())
print()
print('Columnas finales:')
print(df.columns.tolist())
print()
print('Tipos:')
print(df.dtypes)

In [ ]:
# Guardar el dataset limpio
df.to_csv('train_clean.csv', index=False)
print('Guardado como train_clean.csv')